# ModelForge Lite — Phase 1: Environment + Dataset Setup

In [ ]:
!pip install -q datasets huggingface_hub pandas

## 1. Log in to Hugging Face



In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Load the Bitext Customer Support dataset

This is a ready-made, public dataset of customer support questions and answers across common intents (refunds, order status, account issues, etc). No scraping or manual collection needed.

In [ ]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
print(ds)
print(ds["train"][0])

## 3. Explore the data

Look at the intent categories available — we'll narrow to a handful to keep the project focused and explainable.

In [ ]:
import pandas as pd

df = ds["train"].to_pandas()
print(df.columns.tolist())
print(df["intent"].value_counts().head(20))

## 4. Narrow to a focused subset of intents


In [ ]:
selected_intents = [
    "cancel_order",
    "track_order",
    "get_refund",
    "change_order",
    "delivery_options",
    "payment_issue",
]

subset = df[df["intent"].isin(selected_intents)].reset_index(drop=True)
print(f"Subset size: {len(subset)} rows")
subset.head()

## 5. Building the splits

- `train.csv` — used for LoRA fine-tuning (Phase 5)
- `knowledge_base.csv` — a small set of reference answers used for RAG retrieval (Phase 4)
- `eval.csv` — held-out questions, never seen during training or used to build the knowledge base. This is what all four variants get scored on (Phase 7).

In [ ]:
subset = subset.sample(frac=1, random_state=42).reset_index(drop=True)

n_eval = 40
n_kb = 200

eval_df = subset.iloc[:n_eval]
kb_df = subset.iloc[n_eval:n_eval + n_kb]
train_df = subset.iloc[n_eval + n_kb:]

print(f"train: {len(train_df)}  kb: {len(kb_df)}  eval: {len(eval_df)}")

train_df.to_csv("train.csv", index=False)
kb_df.to_csv("knowledge_base.csv", index=False)
eval_df.to_csv("eval.csv", index=False)

## 6. Pushed the splits to our Hugging Face Hub as a dataset

This keeps the data in the cloud so later notebooks (RAG, fine-tuning, eval) can just download it — nothing large stored on our Mac.

In [ ]:
from huggingface_hub import HfApi, create_repo

HF_USERNAME = "YOUR_HF_USERNAME"  # replaced with our hugging face username
repo_id = f"{HF_USERNAME}/modelforge-lite-support-data"

create_repo(repo_id, repo_type="dataset", exist_ok=True)

api = HfApi()
for fname in ["train.csv", "knowledge_base.csv", "eval.csv"]:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=fname,
        repo_id=repo_id,
        repo_type="dataset",
    )

print(f"Uploaded. View at: https://huggingface.co/datasets/{repo_id}")

## Checklist

- [ ] Dataset loaded and explored
- [ ] Narrowed to 5-6 focused intents
- [ ] Split into train / knowledge_base / eval
- [ ] Pushed to our Hugging Face Hub as a dataset
